In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import cell2cell as c2c
import liana as li
import decoupler as dc

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# DIRECTORY SETUP
# ---------------------------------------------------------
INPUT_DIR = './results/tensor_outputs/'
OUTPUT_GSEA = './results/figures/gsea/'
OUTPUT_PROGENY = './results/figures/progeny/'

os.makedirs(OUTPUT_GSEA, exist_ok=True)
os.makedirs(OUTPUT_PROGENY, exist_ok=True)

# ---------------------------------------------------------
# 1. GSEA ANALYSIS FUNCTION
# ---------------------------------------------------------
def run_and_plot_gsea(factors, output_dir, pathwaydb='KEGG', organism='human'):
    """
    Perform Gene Set Enrichment Analysis (GSEA) on the Ligand-Receptor loadings
    extracted from the tensor factorization, and generate a dotplot.
    """
    print(f"[*] Initializing GSEA using {pathwaydb} database for {organism}...")

    # Extract Ligand-Receptor pairs from LIANA consensus resource
    lr_pairs = li.resource.select_resource('consensus')
    lr_list = ['^'.join(row) for idx, row in lr_pairs.iterrows()]

    # Generate gene sets formatted for cell2cell
    lr_set = c2c.external.generate_lr_geneset(
        lr_list,
        complex_sep='_',
        lr_sep='^',
        organism=organism,
        pathwaydb=pathwaydb,
        readable_name=True,
        output_folder=output_dir
    )

    # Run GSEA
    print("[*] Running GSEA with 999 permutations (This may take a moment)...")
    lr_loadings = factors['Ligand-Receptor Pairs']
    pvals, scores, gsea_df = c2c.external.run_gsea(
        loadings=lr_loadings,
        lr_set=lr_set,
        output_folder=output_dir,
        weight=1,
        min_size=15,
        permutations=999,
        processes=6,
        random_state=6,
        significance_threshold=0.05
    )

    # Generate GSEA Dotplot
    print("[*] Generating GSEA Dotplot...")
    dotplot_file = os.path.join(output_dir, f'GSEA_Dotplot_{pathwaydb}.pdf')
    c2c.plotting.pval_plot.generate_dot_plot(
        pval_df=pvals,
        score_df=scores,
        significance=0.05,
        xlabel='',
        ylabel=f'{pathwaydb} Annotations',
        cbar_title='NES',
        cmap='PuOr',
        figsize=(5, 12),
        label_size=16,
        title_size=16,
        tick_size=12,
        filename=dotplot_file
    )
    print(f"  -> GSEA Dotplot saved to {dotplot_file}")

    return pvals, scores

# ---------------------------------------------------------
# 2. PROGENy PATHWAY INFERENCE FUNCTIONS
# ---------------------------------------------------------
def run_progeny_analysis(factors, output_dir):
    """
    Infer signaling pathway activities using PROGENy and the Multivariate Linear Model (MLM).
    Generates a global Clustermap of pathway activities across all factors.
    """
    print("[*] Initializing PROGENy pathway activity inference...")

    # Load PROGENy network weights
    net = dc.get_progeny(organism='human', top=5000)
    lr_pairs = li.resource.select_resource('consensus')

    # Map LIANA ligand-receptor pairs to PROGENy networks
    lr_progeny = li.rs.generate_lr_geneset(lr_pairs, net, lr_sep="^")

    # Run Multivariate Linear Model (MLM)
    print("[*] Estimating pathway activities via MLM...")
    lr_loadings = factors['Ligand-Receptor Pairs']
    estimate, pvals = dc.run_mlm(
        mat=lr_loadings.transpose(),
        net=lr_progeny,
        source="source",
        target="interaction",
        use_raw=False
    )

    # Generate Global PROGENy Clustermap
    print("[*] Generating Global PROGENy Clustermap...")
    clustermap_file = os.path.join(output_dir, 'PROGENy_Global_Clustermap.pdf')

    cm = sns.clustermap(
        estimate,
        xticklabels=estimate.columns,
        cmap='coolwarm',
        z_score=4,
        figsize=(10, 8)
    )
    cm.savefig(clustermap_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  -> PROGENy Clustermap saved to {clustermap_file}")

    return estimate

def export_factor_progeny_barplots(estimate_df, output_dir, num_factors=12, top_n=10):
    """
    Generate and save individual PROGENy pathway activity barplots for each factor.
    """
    print(f"\n[*] Generating detailed PROGENy Barplots for {num_factors} factors...")

    for i in range(1, num_factors + 1):
        factor_name = f'Factor {i}'
        file_name = os.path.join(output_dir, f"PROGENy_Barplot_F{i}.pdf")

        try:
            # Generate barplot using decoupler
            dc.plot_barplot(
                acts=estimate_df,
                contrast=factor_name,
                vertical=True,
                cmap='coolwarm',
                top=top_n,
                figsize=(5, 8)
            )
            # Decoupler plot_barplot uses the current matplotlib axis, so we save it directly
            plt.savefig(file_name, dpi=300, bbox_inches='tight')
            plt.close()

        except Exception as e:
            print(f"     [!] Error plotting {factor_name}: {e}")

    print(f"  -> All individual barplots saved to {output_dir}")

# =========================================================
# EXECUTION PIPELINE
# =========================================================
print("=== 1. LOAD TENSOR FACTORS ===")
factors = c2c.io.load_tensor_factors(os.path.join(INPUT_DIR, 'Loadings.xlsx'))

print("\n=== 2. KEGG PATHWAY ENRICHMENT (GSEA) ===")
# Runs GSEA and saves the NES Dotplot
pval_df, score_df = run_and_plot_gsea(factors, OUTPUT_GSEA, pathwaydb='KEGG')

print("\n=== 3. PROGENy PATHWAY ACTIVITY INFERENCE ===")
# Runs MLM and saves the Global Clustermap
progeny_estimate = run_progeny_analysis(factors, OUTPUT_PROGENY)

print("\n=== 4. SPECIFIC PROGENy BARPLOTS ===")
# Saves 12 individual barplots (PDF) for detailed factor analysis
export_factor_progeny_barplots(progeny_estimate, OUTPUT_PROGENY, num_factors=12, top_n=10)

print("\n[*] DOWNSTREAM BIOLOGICAL VALIDATION COMPLETED SUCCESSFULLY.")